In [7]:
from pathlib import Path
import pandas as pd


# ============================================================
# SETTINGS
# ============================================================

DATA_DIR = Path(".")
UNS_DIR = DATA_DIR / "UNSUPERVISED"

TRANSITIONS = ["AB", "AC", "BCb"]

FEATURES = [30, 20, 12]


# ============================================================
# LOAD + MATCH
# ============================================================

results = []


for transition in TRANSITIONS:

    transition_dir = UNS_DIR / transition

    uns_path = transition_dir / f"{transition}_UNS_summary.csv"
    std_path = transition_dir / f"{transition}_UNS_STD_summary.csv"

    if not uns_path.exists():
        print(f"BRAK: {uns_path}")
        continue

    if not std_path.exists():
        print(f"BRAK: {std_path}")
        continue

    # --------------------------------------------------------
    # LOAD
    # --------------------------------------------------------

    uns = pd.read_csv(uns_path)
    std = pd.read_csv(std_path)

    print(
        f"{transition}: "
        f"UNS={len(uns)}, STD={len(std)}"
    )

    # --------------------------------------------------------
    # tylko interesujące liczby cech
    # --------------------------------------------------------

    uns = uns[
        uns["n_features"].isin(FEATURES)
    ].copy()

    std = std[
        std["n_features"].isin(FEATURES)
    ].copy()

    # --------------------------------------------------------
    # KLUCZ DOPASOWANIA
    #
    # STD ma wybrany stride, więc:
    #
    # transition
    # model
    # n_features
    # stride
    #
    # --------------------------------------------------------

    KEYS = [
        "model",
        "n_features",
        "stride",
    ]

    uns_selected = uns[
        KEYS + [
            "runtime_model",
            "runtime_total",
        ]
    ].rename(
        columns={
            "runtime_model": "runtime_model_UNS",
            "runtime_total": "runtime_total_UNS",
        }
    )

    std_selected = std[
        KEYS + [
            "runtime_model",
            "runtime_total",
        ]
    ].rename(
        columns={
            "runtime_model": "runtime_model_STD",
            "runtime_total": "runtime_total_STD",
        }
    )

    # --------------------------------------------------------
    # MATCH
    # --------------------------------------------------------

    matched = pd.merge(
        std_selected,
        uns_selected,
        on=KEYS,
        how="left",
    )

    # dodaj przejście
    matched.insert(
        0,
        "transition",
        transition,
    )

    results.append(matched)


# ============================================================
# COMBINE
# ============================================================

if not results:
    raise RuntimeError(
        "Nie znaleziono danych do połączenia."
    )

comparison = pd.concat(
    results,
    ignore_index=True,
)


# ============================================================
# ORDER COLUMNS
# ============================================================

comparison = comparison[
    [
        "transition",
        "model",
        "n_features",
        "stride",
        "runtime_model_UNS",
        "runtime_model_STD",
        "runtime_total_UNS",
        "runtime_total_STD",
    ]
]


# ============================================================
# SORT
# ============================================================

comparison = comparison.sort_values(
    [
        "transition",
        "model",
        "n_features",
        "stride",
    ]
).reset_index(drop=True)


# ============================================================
# PRINT
# ============================================================

print("\n")
print("=" * 120)
print("UNS vs STD — RUNTIME")
print("=" * 120)

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}",
    )
)


# ============================================================
# SAVE
# ============================================================

output_path = (
    DATA_DIR
    / "UNSUPERVISED"
    / "UNS_vs_STD_runtime.csv"
)

comparison.to_csv(
    output_path,
    index=False,
)

print("\n")
print(f"Zapisano:")
print(output_path)

AB: UNS=127, STD=24
AC: UNS=105, STD=22
BCb: UNS=111, STD=24


UNS vs STD — RUNTIME
transition           model  n_features  stride  runtime_model_UNS  runtime_model_STD  runtime_total_UNS  runtime_total_STD
        AB   Agglomerative          12      50              3.406              3.106              5.397              4.687
        AB   Agglomerative          20      50              2.879              4.697              4.364              6.778
        AB   Agglomerative          30      50              2.878              3.677              4.484              5.353
        AB     BayesianGMM          12    1000                0.2              0.145              0.742              0.571
        AB     BayesianGMM          20    1000              0.181               0.14              0.577              0.596
        AB     BayesianGMM          30    1000              0.195              0.152              0.703               0.64
        AB           Birch          12      50         